In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("miadul/dna-classification-dataset")

print("Path to dataset files:", path)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|█████████████████████████████████████████| 140k/140k [00:00<00:00, 378kB/s]

Extracting files...
Path to dataset files: /Users/dinarabbasli/.cache/kagglehub/datasets/miadul/dna-classification-dataset/versions/1


In [8]:
import numpy as np
import os
import pandas as pd
path = "/Users/dinarabbasli/.cache/kagglehub/datasets/miadul/dna-classification-dataset/versions/1"
# Qovluqdakı CSV faylının adını tapırıq (adətən bir ədəd olur)
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
df = pd.read_csv(os.path.join(path, csv_files[0]))

In [9]:
df

,Sample_ID,Sequence,GC_Content,AT_Content,Sequence_Length,Num_A,Num_T,Num_C,Num_G,kmer_3_freq,Mutation_Flag,Class_Label,Disease_Risk
0,SAMPLE_1,CTTTCGGGATACTTTTGGGATGGTCTTGGTCAAGGGTTTTAGCCCG...,50.0,50.0,100,22,28,19,31,0.986,0,Bacteria,High
1,SAMPLE_2,TTGACCAAATTTGATTGGAAGTGGTAAGCGCGTATTCCTAGCATCA...,45.0,55.0,100,27,28,22,23,0.486,1,Virus,Medium
2,SAMPLE_3,GCGTGAGTTCTAATTTAAAAAGTCGTAACACGTACCCCGGCGTGTA...,51.0,49.0,100,26,23,30,21,0.367,1,Bacteria,Low
3,SAMPLE_4,ACTACGCGGACAAGAACCAACAGAACCTGGTTTTCGCAAGGGAGTG...,55.0,45.0,100,28,17,23,32,0.404,0,Human,Medium
4,SAMPLE_5,TTCAATGCAGATTGAAAGTTACTTTCATCTGCCCTATGGGTCCCTT...,46.0,54.0,100,24,30,25,21,0.818,0,Human,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,SAMPLE_2996,GATCAGCCCATACACCAAATCAATTGCATACATGTCCGATGTAACA...,46.0,54.0,100,30,24,27,19,0.786,1,Plant,Medium
2996,SAMPLE_2997,TGTTGTGTGTCTGATGATAGGTCATACCGCCTCGAAACATCACCAT...,49.0,51.0,100,28,23,24,25,0.831,0,Plant,Medium
2997,SAMPLE_2998,GACCCACTAAAAGTCTTCGTCTCCTTCCGATGGGAATTTTCGCCGA...,53.0,47.0,100,21,26,30,23,0.140,0,Virus,Medium
2998,SAMPLE_2999,CCAAAGGATATCTGTAATTGTTGCAGCGCCCCTACAATTTGAGCAC...,46.0,54.0,100,26,28,25,21,0.685,0,Plant,Medium


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Sample_ID        3000 non-null   object 
 1   Sequence         3000 non-null   object 
 2   GC_Content       3000 non-null   float64
 3   AT_Content       3000 non-null   float64
 4   Sequence_Length  3000 non-null   int64  
 5   Num_A            3000 non-null   int64  
 6   Num_T            3000 non-null   int64  
 7   Num_C            3000 non-null   int64  
 8   Num_G            3000 non-null   int64  
 9   kmer_3_freq      3000 non-null   float64
 10  Mutation_Flag    3000 non-null   int64  
 11  Class_Label      3000 non-null   object 
 12  Disease_Risk     3000 non-null   object 
dtypes: float64(3), int64(6), object(4)
memory usage: 304.8+ KB


In [13]:
df["Class_Label"].value_counts()


Class_Label
Bacteria    761
Human       749
Plant       747
Virus       743
Name: count, dtype: int64

In [19]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [16]:
pip install tensorflow


  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 MB 6.0 MB/s  0:00:59m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 5.1 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 4.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 5.0 MB/s  0:00:00
Using cached wheel-0.46.3-py3-none-any.whl (30 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 5.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 5.5 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [tensorflow]0 [tensorflow]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
y=df['label'] = df['Class_Label'].apply(lambda x: 1 if x.lower() == 'Human' else 0)

In [38]:
X = df['Sequence'].astype(str).values

In [30]:
# DNA herfleri reqemleree cevirmek bir nov DNT dilini riyazioyyat dilene keciirik herfleri analmir cunki
token = Tokenizer(char_level=True)
token.fit_on_texts(X)
X = tokenizer.texts_to_sequences(X)

In [32]:
# butun Dnalari eyni uzunluqga getiririk melsen 150 edek
max_len = 150 
X = pad_sequences(X, maxlen=max_len)

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42)


In [53]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding


vocab_size = len(tokenizer.word_index) + 1 # Hərflərin sayı (A, T, G, C + padding)
max_len = 150 # teyin elediyimiz uzunluq

model = Sequential([
    # Embedding qatı hərfləri (1, 2, 3...) vektorlara çevirir
    Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len),
    
    # 50 unit olsun
    SimpleRNN(50, activation='tanh'), 
    
    #output-- human (1) və ya bakteriya (0) olduğunu tapmaq ucun
    Dense(1, activation='sigmoid') 
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [60]:
rnn_model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=16, input_length=max_len),
    SimpleRNN(64), # Sadə RNN qatı
    Dense(1, activation='sigmoid')
])
rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [62]:
rnn_model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1)

Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7454 - loss: 0.5746
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7333 - loss: 0.5742
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7475 - loss: 0.5656
Epoch 4/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7475 - loss: 0.5644
Epoch 5/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7475 - loss: 0.5631


In [63]:

lstm_model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=16, input_length=max_len),
    LSTM(64), # LSTM qatı
    Dense(1, activation='sigmoid')
])
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])



lstm_model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1)


Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7400 - loss: 0.5862
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7475 - loss: 0.5697
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7475 - loss: 0.5674
Epoch 4/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7475 - loss: 0.5642
Epoch 5/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7475 - loss: 0.5671


In [65]:
##resultlari muqayise edek
rnn_acc = rnn_model.evaluate(X_test, y_test, verbose=0)[1]
lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)[1]

print(f"\nSimple RNN accuracy: {rnn_acc:.2%}")
print(f"LSTM accuracy: {lstm_acc:.2%}")
# insan  ve ya bakterita olduğunu 76% ehtimalla duz tapir
##her iki modelde eyni deiqiqlik oldu


Simple RNN accuracy: 76.17%
LSTM accuracy: 76.17%
